In [4]:
%run common_setup.ipynb


In [ ]:
#from common_setup import *

input_dir  = 'ALL PREPROCESSED'
output_dir = 'features_d1_d2'
bands = ['D1', 'D2']

os.makedirs(output_dir, exist_ok=True)
writers = {ch: open(os.path.join(output_dir, f'{ch}_features.csv'), 'w') for ch in EEG_CHANNELS}
hdr = make_header(bands)
for w in writers.values():
    w.write(hdr)

for subject in range(1, 29):
    S = f'S{subject:02d}'
    for game in range(1, 5):
        G = f'G{game}'
        val, aro = get_labels(S, G)
        if val is None: continue

        path = os.path.join(input_dir, f'{S}{G}AllChannels.csv')
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        for ch in EEG_CHANNELS:
            if ch not in df.columns: continue
            signal = df[ch].values
            details = pywt.wavedec(signal, 'db2', level=4)[1:3]
            feats = feats_from_details(details)
            line = f'{S},{G},{val},{aro},' + ','.join(map(str, feats)) + '\n'
            writers[ch].write(line)

for w in writers.values(): w.close()
